# 05 - Hyperparameter Tuning and Regularization (Leakage-Safe)

Uses file-level split windows from notebook 02 and evaluates Transformer hyperparameter/regularization configs on raw and quantized settings.
Each run caps training windows to keep per-config runtime bounded.


In [ ]:
# Optional for Colab
# !pip install -q torch pandas numpy


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

PROJECT_OUTPUT_DIR = Path('/content/project_outputs')
CACHE_DIR = PROJECT_OUTPUT_DIR / 'cache'
TABLE_DIR = PROJECT_OUTPUT_DIR / 'tables'

with open('/content/sequence_cache_256seq.json', 'r') as f:
    cache = json.load(f)

V = len(cache['vocab'])
sw = cache['split_windows']
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [3]:
def to_loader(X, y, batch_size=64, shuffle=True):
    X_t = torch.tensor(np.array(X), dtype=torch.long)
    y_t = torch.tensor(np.array(y), dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]


class TransformerNextToken(nn.Module):
    def __init__(self, vocab_size, emb_dim=192, nhead=6, num_layers=3, ff_mult=4, dropout=0.2, max_len=512):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.pos = PositionalEncoding(emb_dim, max_len=max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=emb_dim * ff_mult,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        h = self.emb(x)
        h = self.pos(h)
        h = self.enc(h)
        h = self.drop(h[:, -1, :])
        return self.fc(h)


def run_config(setting_name, split_key, cfg):
    trX, trY = sw[split_key]['train_X'], sw[split_key]['train_y']
    vaX, vaY = sw[split_key]['val_X'], sw[split_key]['val_y']

    max_train_windows = int(cfg.get('max_train_windows', 4000))
    max_val_windows = int(cfg.get('max_val_windows', max_train_windows // 2))
    trX, trY = trX[:max_train_windows], trY[:max_train_windows]
    vaX, vaY = vaX[:max_val_windows], vaY[:max_val_windows]

    tr = to_loader(trX, trY, batch_size=cfg['batch_size'], shuffle=True)
    va = to_loader(vaX, vaY, batch_size=cfg['batch_size'], shuffle=False)

    seq_len = len(trX[0]) if trX else cache.get('seq_len', 128)
    model = TransformerNextToken(
        vocab_size=V,
        emb_dim=cfg['emb_dim'],
        nhead=cfg['nhead'],
        num_layers=cfg['num_layers'],
        ff_mult=cfg['ff_mult'],
        dropout=cfg['dropout'],
        max_len=max(512, seq_len),
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, cfg['epochs']))
    crit = nn.CrossEntropyLoss()

    best = np.inf
    no_imp = 0
    best_epoch = 0
    for ep in range(1, cfg['epochs'] + 1):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()

        model.eval()
        vals = []
        with torch.no_grad():
            for xb, yb in va:
                xb, yb = xb.to(device), yb.to(device)
                vals.append(crit(model(xb), yb).item())
        v = float(np.mean(vals))

        if v < best - 1e-5:
            best = v
            best_epoch = ep
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= cfg['patience']:
                break

    return {
        'setting': setting_name,
        **cfg,
        'architecture': 'transformer_custom',
        'n_train_windows': len(trX),
        'n_val_windows': len(vaX),
        'best_epoch': best_epoch,
        'best_val_loss': best,
    }


In [5]:
configs = [
    {
        'config_id': 't1',
        'emb_dim': 192,
        'nhead': 6,
        'num_layers': 3,
        'ff_mult': 4,
        'dropout': 0.20,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'epochs': 10,
        'patience': 3,
        'batch_size': 64,
        'max_train_windows': 10000,
        'max_val_windows': 2000,
    },
    {
        'config_id': 't2',
        'emb_dim': 192,
        'nhead': 6,
        'num_layers': 3,
        'ff_mult': 4,
        'dropout': 0.30,
        'lr': 8e-4,
        'weight_decay': 5e-4,
        'epochs': 10,
        'patience': 3,
        'batch_size': 64,
        'max_train_windows': 10000,
        'max_val_windows': 2000,
    },
    {
        'config_id': 't3',
        'emb_dim': 256,
        'nhead': 8,
        'num_layers': 4,
        'ff_mult': 4,
        'dropout': 0.25,
        'lr': 6e-4,
        'weight_decay': 1e-3,
        'epochs': 10,
        'patience': 3,
        'batch_size': 48,
        'max_train_windows': 10000,
        'max_val_windows': 2000,
    },
]

rows = []
for cfg in configs:
    rows.append(run_config('quantized_time', 'quant', cfg))

res = pd.DataFrame(rows).sort_values(['setting', 'best_val_loss'])
display(res)


/tmp/ipykernel_3964/2253302767.py:35: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


,setting,config_id,emb_dim,nhead,num_layers,ff_mult,dropout,lr,weight_decay,epochs,patience,batch_size,max_train_windows,max_val_windows,architecture,n_train_windows,n_val_windows,best_epoch,best_val_loss
1,quantized_time,t2,192,6,3,4,0.30,0.0008,0.0005,10,3,64,10000,2000,transformer_custom,10000,2000,7,3.868149
2,quantized_time,t3,256,8,4,4,0.25,0.0006,0.0010,10,3,48,10000,2000,transformer_custom,10000,2000,4,3.909982
0,quantized_time,t1,192,6,3,4,0.20,0.0010,0.0001,10,3,64,10000,2000,transformer_custom,10000,2000,4,3.910387


In [8]:
res.to_csv('05_hparam_tuning_results.csv', index=False)
res.groupby('setting', as_index=False).first().to_csv('05_best_config_by_setting.csv', index=False)
print('Saved notebook 05 outputs.')


Saved notebook 05 outputs.
